## Setup

In [ ]:
import numpy as np
import pandas as pd
import pydot

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, precision_score, balanced_accuracy_score, accuracy_score, recall_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE, RFECV

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from catboost import CatBoostClassifier

from collections import deque


In [ ]:
import logging
import sys
from datetime import datetime

# Configure logging
log_filename = f"training_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler(sys.stdout)  # optional: show logs live in notebook
    ]
)

logger = logging.getLogger(__name__)
logger.info("==== Test logging setup ====")


In [ ]:
# define constants according to your dataset

DATASET = "sample.csv"
COLUMN_NAME_LOS = "DurationHospStayDay"
COLUMN_NAME_ADMISSION_DATE = "DateAdmission"
COLUMN_NAME_TARGET = "Outcome"

In [ ]:
df = pd.read_csv('../../../../data/' + DATASET, low_memory=False)
df = df.drop(columns=[COLUMN_NAME_LOS])
df.drop(COLUMN_NAME_ADMISSION_DATE, axis=1, inplace=True)

In [ ]:
# Example: binary outcome
X = df.drop(COLUMN_NAME_TARGET, axis=1)
y = df[COLUMN_NAME_TARGET]

In [ ]:
label_encoder = LabelEncoder()
label_mapping = {}
for column in X.columns:
  if X[column].dtype == 'object':
    X[column] = X[column].astype(str)
    X[column] = label_encoder.fit_transform(X[column])
    label_mapping[column] = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

logger.info(label_mapping.keys())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=123
)

## Scoring

In [ ]:
scoring = {
    "f1": make_scorer(f1_score, average="binary", pos_label="Dead"),
    "roc_auc": make_scorer(roc_auc_score, needs_proba=True),
    "precision": make_scorer(precision_score, pos_label="Dead"),
    "accuracy": make_scorer(accuracy_score),
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "recall": make_scorer(recall_score, pos_label="Dead")
}

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    res = {
        "f1" : f1_score(y_test, y_pred, average="binary", pos_label="Dead"),
        "roc_auc" : roc_auc_score(y_test, y_pred_proba),
        "precision" : precision_score(y_test, y_pred, pos_label="Dead"),
        "accuracy" : accuracy_score(y_test, y_pred),
        "balanced_accuracy" : balanced_accuracy_score(y_test, y_pred),
        "recall" : recall_score(y_test, y_pred, pos_label="Dead")
    }
    return res

## Models

In [ ]:
models = [{
        "model": LogisticRegression(max_iter=3000),
        "model_name": "Logistic Regression",
        "hyperparameters": {
            "model__C": [0.01, 0.1, 1, 10, 100],   # regularization strength
            "model__penalty": ["l1", "l2", "elasticnet"],
            "model__l1_ratio": [0, 0.5, 1]         # only used if elasticnet
        }
    },
    {
        "model": GaussianNB(),
        "model_name": "Gaussian Naive Bayes",
        "hyperparameters": {
            "model__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]   # handles numerical stability
        }
    },
    {
        "model": DecisionTreeClassifier(),
        "model_name": "Decision Tree Classifier",
        "hyperparameters": {
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 5, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [None, "sqrt", "log2"]
        }
    },
    {
        "model": RandomForestClassifier(),
        "model_name": "Random Forest Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": SVC(probability=True),
        "model_name": "Support Vector Classifier",
        "hyperparameters": {
            "model__C": [0.01, 0.1, 1, 10, 100],
            "model__kernel": ["linear", "rbf", "poly"],
            "model__degree": [2, 3, 4],   # only for poly
            "model__gamma": ["scale", "auto"]
        }
    },
    {
        "model": ExtraTreesClassifier(criterion="entropy",random_state=123),
        "model_name": "Extra Trees Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": KNeighborsClassifier(n_neighbors=3,metric='minkowski',p=2),
        "model_name": "K-Neighbors Classifier",
        "hyperparameters": {
            "model__n_neighbors": [3, 5, 7, 11],
            "model__weights": ["uniform", "distance"],
            "model__metric": ["minkowski", "manhattan"],
            "model__p": [1, 2]   # 1=Manhattan, 2=Euclidean
        }
    },
    {
        "model": GradientBoostingClassifier(),
        "model_name": "Gradient Boosting Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [3, 5, 7],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__subsample": [0.6, 0.8, 1.0],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": CatBoostClassifier(verbose=False),
        "model_name": "CatBoost Classifier",
        "hyperparameters": {
            "model__iterations": [200, 500, 1000],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__depth": [4, 6, 8, 10],
            "model__l2_leaf_reg": [1, 3, 5, 7]
        }
    }]

## Tree Processing

In [ ]:
def getRealNodes(nodes):
    real_nodes = []
    for node in nodes:
        if node.get_attributes().get("label") is not None:
            real_nodes.append(node)
    return real_nodes

In [ ]:
def jsonify_nodes(root_node_name, node_list, edge_list):
    """
    Traverses a pydot graph from a root node and returns a list of dictionaries 
    with each node's label and level.

    Args:
        root_node_name (str): The name of the starting (root) node.
        node_list (list): A list of pydot Node objects.
        edge_list (list): A list of pydot Edge objects.

    Returns:
        list: A list of dictionaries, each containing a node's 'label' and 'level'.
    """
    # 1. Build an adjacency list for efficient neighbor lookup
    adj = {node.get_name(): [] for node in node_list}
    for edge in edge_list:
        source = edge.get_source()
        destination = edge.get_destination()
        if source in adj:
            adj[source].append(destination)
    
    # 2. Map node names to their labels
    # Use .get_label() but handle the case where it returns a name, which occurs
    # if no explicit label is set.
    name_to_label_map = {
        node.get_name(): node.get_attributes().get("label")
        if node.get_label() != node.get_name() else node.get_name()
        for node in node_list
    }

    # 3. Prepare data structures for BFS
    queue = deque([(root_node_name, 0)])  # Store tuples of (node_name, level)
    visited = {root_node_name}
    result_list = []

    # 4. Perform the BFS traversal
    while queue:
        current_node_name, level = queue.popleft()
        
        # Get the label from the map we created
        current_label = name_to_label_map.get(current_node_name, current_node_name)
        
        # Add the current node's label and level to the result
        result_list.append({
            "label": current_label,
            "level": level
        })

        # Enqueue all unvisited neighbors
        for neighbor_name in adj.get(current_node_name, []):
            if neighbor_name not in visited:
                visited.add(neighbor_name)
                queue.append((neighbor_name, level + 1))
    
    return result_list

In [ ]:
def tabulate_nodes(json_nodes):
    for node in json_nodes:
        node['label'] = node['label'].split(" ")[0].replace('"', '')
        
    return pd.DataFrame(json_nodes)

test_list = [{
    "label": "BW <= ashdjkasdhk"
}]

tabulate_nodes(test_list)

In [ ]:
def compute_rankings(tree_count):
    df_list = []
    for i in range(tree_count):
        graph = pydot.graph_from_dot_file(f'Decision paths for Tree {i+1}.dot')
        real_nodes = getRealNodes(graph[0].get_nodes())
        jsonified_nodes = jsonify_nodes(real_nodes[0].get_name(), graph[0].get_nodes(), graph[0].get_edges())
        df_list.append(tabulate_nodes(jsonified_nodes))
        
    aggregated_df = pd.concat(df_list, ignore_index=True)
    return aggregated_df

In [ ]:
def count_by_label(aggregated_df):
    label_counts = (
        aggregated_df
        .groupby(['level', 'label'])
        .size()
        .reset_index(name='count')
        .sort_values(by=['level', 'count'], ascending=[True, False])
    )
    return label_counts

In [ ]:
def getScore(levels:pd.Series):
    max_level = levels.max()
    logger.info(f"Max level: {max_level}")
    scores = []
    for level in levels:
        # The lower the level, the higher the score
        scores.append(max_level - level)
    return scores

In [ ]:
def label_counts_with_score(label_counts):
    label_counts_with_score_df = label_counts.copy()
    label_counts_with_score_df['score'] = (label_counts_with_score_df['count'] * getScore(label_counts_with_score_df['level']))
    label_counts_with_score_df = label_counts_with_score_df.drop(columns=['count', 'level'])
    label_counts_with_score_df = label_counts_with_score_df.groupby(['label'])['score'].sum().reset_index()
    label_counts_with_score_df = label_counts_with_score_df.sort_values(by=['score'], ascending=False)
    return label_counts_with_score_df

## Feature name correction

In [ ]:
def rectify_feature_names(ranked_features):
    for i, feature in enumerate(ranked_features):
        if feature == "AtSteroid":
            ranked_features[i] = "AtSteroid Dose_term"
        if feature == "Intrapartum":
            ranked_features[i] = "Intrapartum Antibiotic_term"
    ranked_features.remove("friedman_mse")
    return ranked_features

## Modeling

In [ ]:
logger.info("Rerun by taking out RFECV from pipeline because NaN encountered during training")

In [ ]:
base_estimator = CatBoostClassifier(verbose=False)

# Define inner CV for RFECV (can be the same as main CV)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# Define scoring
scorer = make_scorer(roc_auc_score, needs_proba=True)

# === Skip this step as this is not how RFECV is usually used ===
# feature_group_list = []
# for i in range(len(X_train.columns)):
#     feature_group = RFE(base_estimator, n_features_to_select=i+1).fit(X_train, y_train)
#     selected_features = X_train.columns[feature_group.support_].tolist()
#     feature_group_list.append(selected_features)
#     logger.info(f"Feature Group {i+1}: {selected_features}")

In [ ]:
rfecv = RFECV(
    estimator=base_estimator,
    step=1,
    cv=inner_cv,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

logger.info("Running RFECV for feature ranking...")
rfecv.fit(X_train, y_train)
logger.info(f"Optimal number of features: {rfecv.n_features_}")

# === Rank features by importance ===
feature_ranks = pd.DataFrame({
    "feature": X_train.columns,
    "ranking": rfecv.ranking_,
    "support": rfecv.support_
}).sort_values("ranking")

logger.info("Feature ranking complete.")
logger.info(f"\n{feature_ranks}")

In [ ]:
# Full pipeline: Oversampling + Classifier
pipeline = ImbPipeline([
    ("ros", SMOTE(random_state=123)),
    ("model", base_estimator)
])

# Hyperparameter grid
param_grid = {
    "model__iterations": [200, 500, 1000],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__depth": [4, 6, 8, 10],
    "model__l2_leaf_reg": [1, 3, 5, 7]
}

# Outer CV (for tuning)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

# === Skip this step as this is not how RFECV is usually used ===
# for feature_group in feature_group_list:
#     logger.info(f"Features: {feature_group}")
#     X_train_sub = X_train[feature_group]
#     # Grid search
#     search = GridSearchCV(
#         estimator=pipeline,
#         param_grid=param_grid,
#         cv=outer_cv,
#         scoring=scorer,
#         n_jobs=-1,
#         verbose=0,
#         error_score='raise'
#     )
    
#     search.fit(X_train_sub, y_train)
    
#     logger.info(f"Best CV score: {search.best_score_}")
#     logger.info(f"Best hyperparameters: {search.best_params_}")

## Train with RFECV results only without exhaustive search

In [ ]:
selected_features = X_train.columns[rfecv.support_]
logger.info(f"Selected Features ({len(selected_features)}): {list(selected_features)}")

In [ ]:
logger.info("Train with structure below:")
logger.info("1. Features selected by RFECV")
logger.info("2. Hyperparameter tuning with GridSearchCV")

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=outer_cv,
    scoring=scorer,
    n_jobs=-1,
    verbose=0,
    error_score='raise'
)

search.fit(X_train[selected_features], y_train)

logger.info(f"Best CV score: {search.best_score_}")
logger.info(f"Best hyperparameters: {search.best_params_}")